In [1]:
import whisper
from pathlib import Path
import os

In [2]:
model = whisper.load_model("medium")

In [ ]:
result=model.transcribe(verbose=True)

TypeError: transcribe() missing 1 required positional argument: 'audio'

In [3]:
def transcribe_folder(folder_path):
    recordings_dir = Path(folder_path)
    results = []

    # 1. Find all audio files (wav, mp3, m4a, etc.)
    audio_extensions = ["*.wav", "*.mp3", "*.m4a", "*.flac"]
    audio_files = []
    for ext in audio_extensions:
        audio_files.extend(recordings_dir.glob(ext))

    if not audio_files:
        print("No audio files found in the directory.")
        return pd.DataFrame()

    print(f"Found {len(audio_files)} files. Starting transcription...")

    # 2. Loop through and transcribe
    for file_path in audio_files:
        try:
            print(f"Processing: {file_path.name}...")
            
            # Transcribe the file
            # Use fp16=False if you are on a CPU or get a warning
            result = model.transcribe(str(file_path))
            
            # Store data in a dictionary
            results.append({
                "filename": file_path.name,
                "transcription": result["text"].strip(),
                "language": result.get("language", "unknown")
            })
            break
            
        except Exception as e:
            print(f"Failed to transcribe {file_path.name}: {e}")
            results.append({
                "filename": file_path.name,
                "transcription": f"ERROR: {str(e)}",
                "language": "N/A"
            })

    df = pd.DataFrame(results)
    return df 

In [4]:
import pandas as pd

In [5]:
if __name__ == "__main__":
    # Specify your recordings folder path
    recordings_path = r"C:\Users\pratik p kakade\recordings" 
    
    # Run the function
    df_results = transcribe_folder(recordings_path)
    
    # Display the result
    print("\nTranscription Results:")
    print(df_results)

Found 20 files. Starting transcription...
Processing: call_recording_01.wav...


c:\Users\pratik p kakade\Desktop\infosys springboard\.venv\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcription Results:
                filename                                      transcription  \
0  call_recording_01.wav  Hello, I'm Sarah Miller. I'm calling to inquir...   

  language  
0       en  


In [6]:
df_results.columns

Index(['filename', 'transcription', 'language'], dtype='object')

In [11]:
a=df_results['transcription'][0]
a

"Hello, I'm Sarah Miller. I'm calling to inquire about the AC7892 air conditioner unit. I saw it on your website and I had a few questions. First, what's the BTU rating? And second, does it come with a remote control or is that sold separately?"

In [12]:



def get_stratified_sample(text, max_tokens=420):
    words = text.split()
    total_words = len(words)
    
    # Approx 0.75 words per token. 420 tokens ~ 315 words.
    sample_size = 100 # words per section
    
    if total_words <= sample_size * 3:
        return text # Return full text if it's already short

    start = " ".join(words[:sample_size])
    middle = " ".join(words[total_words//2 - 50 : total_words//2 + 50])
    end = " ".join(words[-sample_size:])
    
    return f"[START]: {start} ... [MIDDLE]: {middle} ... [END]: {end}"



In [9]:
from groq import Groq

def score_with_groq(transcript):
    # 1. Condense the transcript to fit the token budget
    # condensed_text = get_stratified_sample(transcript)
    condensed_text = transcript

    # 2. Define the scoring rubric
    system_prompt = (
        "You are an expert Quality Assurance analyst. Analyze the call transcript segments provided. "
        "Score the agent on a scale of 1-100 for Empathy, Professionalism, and Compliance. "
        "Compliance requires checking if the agent used a standard greeting and a closing disclaimer. "
        "Return ONLY a JSON object with keys: 'empathy_score', 'professionalism_score', "
        "'compliance_score', 'violations', and 'suggestions'."
    )

    client=Groq(api_key="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")

    try:
        # 3. Call Groq API
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this condensed transcript: {condensed_text}"}
            ],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"}  # Forces Groq to return valid JSON
        )
        
        # Parse and return the result
        return json.loads(chat_completion.choices[0].message.content)

    except Exception as e:
        return {"error": str(e)}

In [13]:
for index, row in df_results.iterrows():
        print(f"Scoring record {index + 1}/{len(df_results)}...")

        
        # Get the score
        print(row['transcription'])

Scoring record 1/1...
Hello, I'm Sarah Miller. I'm calling to inquire about the AC7892 air conditioner unit. I saw it on your website and I had a few questions. First, what's the BTU rating? And second, does it come with a remote control or is that sold separately?


In [14]:
import os
import json
import time
import pandas as pd
from groq import Groq
from pathlib import Path

if __name__ == "__main__":
        
    results_list = []

    print(f"Starting Quality Scoring for {len(df_results)} records...")

    for index, row in df_results.iterrows():
        print(f"Scoring record {index + 1}/{len(df_results)}...")
        
        # Get the score
        score_data = score_with_groq(row['transcription'])
        results_list.append(score_data)
        
        # 2. Rate Limiting: Pause to stay under Groq Free Tier RPM limits
        time.sleep(1.5) 
        break

    # 3. Merge results back into the original DataFrame
    # pd.json_normalize converts the list of dicts into separate columns automatically
    df_scores = pd.json_normalize(results_list)
    df_results = pd.concat([df_results, df_scores], axis=1)

    # 4. Save or View
    print("\nScoring Complete!")
    print(df_results.head())
    # df.to_csv("final_qa_report.csv", index=False)

Starting Quality Scoring for 1 records...
Scoring record 1/1...

Scoring Complete!
                filename                                      transcription  \
0  call_recording_01.wav  Hello, I'm Sarah Miller. I'm calling to inquir...   

  language  empathy_score  professionalism_score  compliance_score  \
0       en              0                      0                 0   

                                          violations  \
0  [No standard greeting by the agent, No closing...   

                                         suggestions  
0  [Agent should respond with a standard greeting...  


In [19]:
df_results.head()

,filename,transcription,language,empathy_score,professionalism_score,compliance_score,violations,suggestions,empathy_score,professionalism_score,compliance_score,violations,suggestions
0,call_recording_01.wav,"Hello, I'm Sarah Miller. I'm calling to inquir...",en,0,0,0,"[No standard greeting, No response from agent,...","[Use a standard greeting, Respond to customer ...",0,0,0,"[No standard greeting from the agent, No closi...",[Agent should respond with a standard greeting...


In [6]:
import mysql.connector

# --- 1. DATABASE CONFIGURATION ---
db_config = {
    "host": "localhost",
    "user": "root",        # Replace with your MySQL username
    "password": "MYSQL", # Replace with your MySQL password
    "database": "quality_auditor"
}

In [5]:
try:
    conn=mysql.connector.connect(**db_config)
    print("db connected")
except mysql.connector.Error as e:
    print(f"Database error: {e}")


db connected


In [6]:
conn.close()

In [4]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv
load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("infosys-sprinboard") 

# Embedding Model 
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\pratik p kakade\Desktop\infosys springboard\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 267.31it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import re

def is_luhn_valid(number):
    """Checks if a string of digits passes the Luhn Algorithm."""
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if len(digits) < 13: return False
    
    # Reverse the digits and apply Luhn logic
    checksum = 0
    reverse_digits = digits[::-1]
    for i, digit in enumerate(reverse_digits):
        if i % 2 == 1:
            digit *= 2
            if digit > 9:
                digit -= 9
        checksum += digit
    return checksum % 10 == 0

def mask_pii(text):
    """Redacts PII with smart validation to avoid false positives."""
    
    # 1. Advanced Credit Card Redaction
    # First, find potential 13-16 digit candidates
    card_candidates = re.findall(r'\b(?:\d[ -]*?){13,19}\b', text)
    for candidate in card_candidates:
        if is_luhn_valid(candidate):
            text = text.replace(candidate, '[VALID_CARD_REDACTED]')

    # 2. Email Addresses (Standard)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    
    # 3. Phone Numbers (Improved to avoid small IDs)
    # Looks for specific phone patterns: (123) 456-7890 or 123-456-7890
    text = re.sub(r'\b(?:\+?\d{1,3}[- ]?)?\(?\d{3}\)?[- ]?\d{3}[- ]?\d{4}\b', '[PHONE]', text)
    
    # 4. SSN
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    
    return text

In [4]:
a=mask_pii("I’m ready to get this subscription started. My card number is 4532 8810 9925 4071, and the expiration date is 03/28 with the CVV 512. For the account records, you can reach me at 555-012-8493 if there are any issues with the setup. Please make sure the digital receipt is sent to my primary email address, which is j.henderson@emailprovider.com. Let me know once the transaction goes through on your end.")
a

'I’m ready to get this subscription started. My card number is [CARD_NUMBER], and the expiration date is 03/28 with the CVV 512. For the account records, you can reach me at [PHONE] if there are any issues with the setup. Please make sure the digital receipt is sent to my primary email address, which is [EMAIL]. Let me know once the transaction goes through on your end.'